# AI Integrations for Developers — Exam

## Instructions

- This notebook is a **template** where you must put your code.  
- You should **fill in all empty variables** and complete the code so that when I download your notebook and click **Run all**, all cells execute correctly and provide the answers.  
- ⚠️ **Do NOT hardcode your API key**. Use Colab environment variables (`%env OPENAI_API_KEY=your_key_here`) and access them in your code.  
- You may **create more cells** if needed. It is recommended that your code is well-structured and split logically into separate cells.  
- The function **`ask_ai(query)`** must be implemented by you. All queries will call this function to check your solution.  
- ✅ **Test cases will be created by me (the instructor).** You are **not allowed to modify, remove, or add to the test cases cell**. Your code must work correctly with the provided test cases.  
- You are **ONLY ALLOWED** to use only the following:  
  - **Models:** OpenAI or Anthropic  
  - **Technologies:** LangChain or vanilla Python code  
  - **Vector Store:** Chroma DB

🚨 **Any student who does not follow the template, does not stick to the required format, or whose code does not execute properly will be disqualified.**


### Important

Fill in **all the variables** in the cell.  
❌ **Do NOT put your API key directly in the code.**  
✅ The cell must be set up to take the API key from the Colab environment variables.


In [147]:
# ================================
# 🔧 RAG Configuration Variables
# ================================

# ⚠️ Do NOT put your API key here directly.
# Make sure you set your API key in Colab like this:
# %env OPENAI_API_KEY=your_key_here

import os
from google.colab import userdata

# API Key (taken from Colab environment variables)
API_KEY = userdata.get("OPENAI_API_KEY")

os.environ["OPENAI_API_KEY"] = API_KEY

# Prompt & Model Settings:

# 1. Generates PDF summary
SYSTEM_MESSAGE_PDF_SUMMARY_GENERATOR = """
<context>
We are building a chatbot to answer users' queries based on information from a PDF file. The PDF file is chunked and stored in a vector database. We need a concise summary that will help an AI assistant optimize user queries for better vectorstore retrieval. The summary will be used in a query optimization step where an AI assistant receives both this summary and a user's original query to generate an optimized search query for the vectorstore.
</context>

<role>
You are an experienced Knowledge Manager specializing in document analysis for RAG systems.

<skills>
- Extract key topics, themes, and subject areas from documents
- Identify important terminology and domain-specific vocabulary
- Create concise overviews that capture document scope without detail
- Structure information for AI query optimization
- Distinguish between high-level concepts and specific details
- Understand how document summaries aid in search query formulation
</skills>

<experience>
- 3+ years working with retrieval-augmented generation systems or AI-powered search
- Background in prompt engineering or AI system optimization
- Knowledge of how document structure affects AI retrieval performance
- Experience preparing content for vector databases and understanding chunking strategies
- Experience creating executive summaries, abstracts, or document overviews
- Understanding of how users formulate queries and search for information
</experience>
</role>

<task>
Create a brief summary of the PDF that includes:
1. Main Topics: What subjects/areas does the document cover?
2. Key Terminology: Important terms, concepts, or vocabulary used
3. Document Structure: Major sections or categories of information
4. Content Types: What kinds of information can users expect to find (procedures, data, policies, numbers, etc.)

<important>
- Focus on WHAT the document contains, not the specific details
- Use terminology from the original document
- Keep it concise
- Structure it to help with query optimization
</important>
</task>

<next>
Output only the summary text without additional formatting or commentary.
</next>
"""

HUMAN_MESSAGE_PDF_SUMMARY_GENERATOR = """
PDF file: {pages}
"""

# 2. Contextualizes Chunks
SYSTEM_MESSAGE_CHUNK_CONTEXTUALIZER = """
<role>
You are a document processing specialist. Your task is to add contextual information to document chunks to improve their retrieval in a vector database.
</role>

<task>
Given a document chunk and a summary of the source document, provide a brief contextual prefix (50-100 tokens) that situates the chunk within the overall document. This context should:
- Identify which section or topic the chunk relates to based on the document summary
- Clarify any references that might be unclear when the chunk stands alone
- Preserve the meaning and searchability of the original content
- Use terminology from the document summary and chunk
<task>

<next>
Output only the contextual prefix, nothing else.
</next>
"""

HUMAN_MESSAGE_CHUNK_CONTEXTUALIZER = """
SUMMARY:\n{document_summary}\n\n
CHUNK:\n{chunk_content}
"""

# 3. Creates optimized query
SYSTEM_MESSAGE_QUERY_OPTIMIZER = """
<context>
Our client hired us to develop a chatbot that answers user questions based on information from their PDF document. The PDF has been processed, chunked, and stored in a vectorstore. Users interact with the chatbot through conversational questions, and the system's effectiveness depends entirely on successfully retrieving the most relevant chunks from the vectorstore. Query optimization is critical because poor retrieval leads to irrelevant or incomplete responses, directly impacting user satisfaction and system performance.
</context>

<role>
You are an Information Architect specializing in query optimization for RAG (Retrieval-Augmented Generation) systems. Your task is to analyze user queries and transform them into optimized search queries that will retrieve the most relevant content from a vectorstore.

<skills>
- Parse user intent from conversational queries
- Identify key concepts, entities, and relationships in unstructured text
- Handle ambiguous, incomplete, or poorly structured user inputs
- Recognize synonyms and related terms that might exist in the vectorstore
- Transform natural language into effective search terms
- Understand vector similarity matching and semantic search principles
- Balance query specificity vs. breadth for optimal retrieval
- Map user concepts to document-specific vocabulary and terminology
- Recognize hierarchical relationships and implicit requirements
- Ensure optimized queries align with vectorstore chunking structure
- Maintain user intent while maximizing retrieval success
</skills>

<experience>
- 3+ years working with search engines, vector databases, or recommendation systems
- Experience with semantic search, embeddings, and similarity matching
- Knowledge of retrieval metrics (precision, recall, relevance scoring)
- Understanding of how document chunking affects search performance
- Experience building or optimizing retrieval-augmented generation systems
- Understanding of how retrieval quality impacts downstream generation
- Knowledge of prompt engineering and context window optimization
- Experience with vector stores (Pinecone, Weaviate, Chroma, etc.)
- Experience building AI-powered applications for business users
- Understanding of how users search for and consume information
</experience>
</role>

<task>
1. Analazy the USER QUERY
1. Study the provided PDF SUMMARY to understand: (1) main topics covered, (2) key terminology used, (3) document structure, and (4) types of information available. This knowledge will guide your query optimization decisions
2. Create one optimized search query that will effectively retrieve relevant content from the vectorstore
</task>

<next>
<important>
- Focus on terms, concepts, and phrases that are likely to match the chunked content while maintaining the user's original intent expressed into the USER QUERY
- Your optimized query should use terminology and concepts present in the source document to ensure successful retrieval from the vectorstore
- Do not include irrelevant keywords that could block matching the user’s intended information
</important>

Output only the optimized search query as a question in the customer's voice. Do not include anything else except the question.
</next>
"""

HUMAN_MESSAGE_QUERY_OPTIMIZER = """
USER QUERY:\n{query}\n\n
PDF SUMMARY:\n{pdf_summary}
"""

# 4. Responds to the user
SYSTEM_MESSAGE_RESPONDER = """
You are a helpful assistant.
<critical_rules>
1. End with a complete sentence without cutting off mid-thought, mid-sentence or mid-paragraph.
2. Limit discussions only to information into the CONTEXT and the CONVERSATION MEMORY.
</critical_rules>
"""

HUMAN_MESSAGE_RESPONDER = """
CONVERSATION MEMORY:\n{conversation_memory}\n\n
Based on the following CONTEXT:\n{context}\n\n
Answer the following QUESTION:\n{query}
"""

MODEL = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-small"

# Chunking Parameters
CHUNK_SIZE = 650
CHUNK_OVERLAP = 80
TOP_N_RESULTS = 3

# Generation Parameters
OUTPUT_LENGTH = 450
TEMPERATURE = 0.1
TOP_P = 0.1
FREQUENCY_PENALTY = 1.0
PRESENCE_PENALTY = 1.0

### Code Organization

Create more cells if needed and put your code in them.  
It is **recommended** that your code is well-structured, split logically, and kept in separate cells for clarity.


### 1. Install Dependencies

In [148]:
!pip install -U langchain langchain-openai langchain-community chromadb pypdf python-dotenv

### 2. Upload PDF to Colab

In [149]:
from google.colab import files

uploaded_file = files.upload()

Saving Exam_Preparation_PDF.pdf to Exam_Preparation_PDF (11).pdf


### 3. Load the PDF

In [150]:
from langchain_community.document_loaders import PyPDFLoader

pdf_path = list(uploaded_file.keys())[0]
loader = PyPDFLoader(pdf_path)

pages = loader.load()

### 4. Chunk the PDF

In [151]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
      chunk_size=CHUNK_SIZE,
      chunk_overlap=CHUNK_OVERLAP,
      separators=[ "\n\n", "\n", ".", "!", "?", ",", ";", ":", "-"],
  )

chunks = text_splitter.split_documents(pages)

Chunks
[Document(metadata={'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 19.5 (Macintosh)', 'creationdate': '2024-10-11T12:21:56-04:00', 'moddate': '2024-10-11T12:22:09-04:00', 'trapped': '/False', 'source': 'Exam_Preparation_PDF (11).pdf', 'total_pages': 68, 'page': 0, 'page_label': '1'}, page_content='1\nOctober 2024 edition\nA quick-start handbook \nfor effective prompts'), Document(metadata={'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 19.5 (Macintosh)', 'creationdate': '2024-10-11T12:21:56-04:00', 'moddate': '2024-10-11T12:22:09-04:00', 'trapped': '/False', 'source': 'Exam_Preparation_PDF (11).pdf', 'total_pages': 68, 'page': 1, 'page_label': '2'}, page_content='2\nWriting effective prompts \nFrom the very beginning, Google Workspace was built to allow you to collaborate in real time with other people. \nNow, you can also collaborate with AI using Gemini for Google Workspace to help boost your productivity and \ncreativity without sacrificing

### 5. Initialize embedding model

In [152]:
from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings(
     model=EMBEDDING_MODEL
  )

### 6. Initialize Conversation Buffer Memory

In [153]:
from langchain.memory import ConversationBufferMemory

memory = ConversationBufferMemory(
    memory_key="conversation_memory",
    return_messages=True,
    output_key="response"
)

### 7. Initialize the Chat Model

In [154]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model=MODEL,
    max_tokens=OUTPUT_LENGTH,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    frequency_penalty=FREQUENCY_PENALTY,
    presence_penalty=PRESENCE_PENALTY,
    streaming=True,
)

### 8. Define function to generate AI responses

In [155]:
from langchain.prompts import ChatPromptTemplate
from langchain.prompts import (
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
)

def generate_ai_response(
    llm,
    system_message,
    human_message,
    **kwargs
):
    prompt = ChatPromptTemplate.from_messages([
        SystemMessagePromptTemplate.from_template(
            system_message
        ),
        HumanMessagePromptTemplate.from_template(
            human_message
        ),
    ])

    messages = prompt.format_messages(**kwargs)

    return llm.invoke(messages).content

### 9. Generate PDF summary

In [156]:
pdf_summary = generate_ai_response(
      llm,
      SYSTEM_MESSAGE_PDF_SUMMARY_GENERATOR,
      HUMAN_MESSAGE_PDF_SUMMARY_GENERATOR,
      pages=pages,
    )

### 10. Initialize and populate Chroma DB using Contextual Retrieval (this will take about 3 minutes)

In [171]:
from langchain_community.vectorstores import Chroma
from langchain.schema import Document


def contextualize_chunk(chunk, document_summary, llm):
    """
    Add contextual information to a chunk using document summary.
    """
    context = generate_ai_response(
        llm,
        SYSTEM_MESSAGE_CHUNK_CONTEXTUALIZER,
        HUMAN_MESSAGE_CHUNK_CONTEXTUALIZER,
        document_summary=document_summary,
        chunk_content=chunk.page_content
    )

    # Prepend context to the chunk content
    contextualized_content = f"{context.strip()} {chunk.page_content}"

    # Create a new document with contextualized content
    return Document(
        page_content=contextualized_content,
        metadata=chunk.metadata
    )

def create_contextualized_vectorstore(pages, chunks, embedding_model, pdf_summary, llm):
    """
    Create a vectorstore with contextualized chunks using document summary.
    """

    print(f"Contextualizing {len(chunks)} chunks using document summary...")

    # Contextualize each chunk using the summary
    contextualized_chunks = []
    for i, chunk in enumerate(chunks):
        print(f"Processing chunk {i+1}/{len(chunks)}")
        contextualized_chunk = contextualize_chunk(chunk, pdf_summary, llm)
        contextualized_chunks.append(contextualized_chunk)

    print("Creating vectorstore with contextualized chunks...")

    # Create vectorstore with contextualized chunks
    vectorstore = Chroma.from_documents(
        documents=contextualized_chunks,
        embedding=embedding_model,
        persist_directory=None,
    )

    return vectorstore

vectorstore = create_contextualized_vectorstore(pages, chunks, embedding_model, pdf_summary, llm)

Contextualizing 257 chunks using document summary...
Processing chunk 1/257


KeyboardInterrupt: 

### 11. Define function to retrieve relevant content from database

In [158]:
def retrieve_relevant_context(
    vectorstore,
    query,
    k=4
):
    results = vectorstore.similarity_search(
        query,
        k=k
    )
    context = '\n'.join(
        result.page_content for result in results
    )

    return context.strip()

## Test Cases (Final Cell)

The final cell must contain your **test cases**.  
When executed, the AI should provide correct answers to the given questions **based on the PDF file**.


### AI Query Function

In this cell, you must implement the function **ask_ai(query)**.  
This function will be the final execution point of your pipeline (RAG / LLM).  


In [168]:
# ================================
# ❓ AI Query Function
# ================================

def ask_ai(query: str):
    """
    This function should execute your final RAG / LLM pipeline.
    Input:
        query (str): The question you want to ask the AI.
    Output:
        str: The AI's answer based on the PDF file.
    """
    # TODO: Implement your final execution logic here
    # Example steps:
    # 1. Retrieve relevant chunks
    # 2. Generate embeddings
    # 3. Call the model with your prompt + retrieved context
    # 4. Return the model's answer

    optimized_query = generate_ai_response(
      llm,
      SYSTEM_MESSAGE_QUERY_OPTIMIZER,
      HUMAN_MESSAGE_QUERY_OPTIMIZER,
      query=query,
      pdf_summary=pdf_summary,
    )

    context = retrieve_relevant_context(
      vectorstore,
      optimized_query,
      TOP_N_RESULTS,
    )

    ai_response = generate_ai_response(
      llm,
      SYSTEM_MESSAGE_RESPONDER,
      HUMAN_MESSAGE_RESPONDER,
      conversation_memory=memory.load_memory_variables({})['conversation_memory'],
      context=context,
      query=query,
    )

    memory.save_context({"input": query}, {"response": ai_response})


    return ai_response

### Test Queries

Use this cell to test your function with different queries.  
The answers must be generated correctly based on the PDF file.  


In [170]:
# ================================
# 🔍 Example Queries for Testing
# ================================

queries = [
    "What is pizza?"
    # "How many words should effective prompts average?",
    # "List the four main areas for effective prompts.",
    # "What does 'persona' mean in prompt writing?",
    # "Name three business roles covered in this guide.",
    # "What is Gemini Advanced?",
]

# Call the AI with each query
for q in queries:
    print(f"Q: {q}")
    print(f"A: {ask_ai(q)}\n")


Q: What is pizza?
A: Pizza is a popular dish that typically consists of a round, flat base of dough topped with various ingredients such as tomato sauce, cheese, meats, vegetables, and herbs. It is baked in an oven and can be served in various styles and sizes. Pizza originated from Italy but has become widely enjoyed around the world with many regional variations.

